# D2 — Data profile and cleaning log
NYC flights 2013 (`nycflights13`). Section numbers match the D2 report: §6 Data profile, §7 Cleaning log.
Working copy only: `data/raw/` is read-only and is never edited.

In [ ]:
import pandas as pd
import numpy as np

SRC = "flights_working.csv"      # working copy (flights merged with airlines); raw files stay untouched
df = pd.read_csv(SRC)
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")

## 6. Data profile (as loaded, before any cleaning)

In [ ]:
def profile(d):
    n = len(d)
    out = []
    for c in d.columns:
        s = d[c]
        nn = int(s.notna().sum())
        if pd.api.types.is_numeric_dtype(s):
            rng = f"{s.min():,.0f} to {s.max():,.0f}"
        else:
            rng = f"{s.nunique():,} distinct"
        out.append((c, str(s.dtype), nn, n - nn, round((n - nn) / n * 100, 2), rng))
    return pd.DataFrame(out, columns=["Column", "Type", "Non-null", "Missing", "Missing %", "Range or distinct values"])

profile(df)

## 7. Cleaning log
### Decision 1 — Merge check (airlines → flights on `carrier`)

In [ ]:
print("carrier->name is 1:1:", df.groupby("carrier")["name"].nunique().max() == 1 and df.groupby("name")["carrier"].nunique().max() == 1)
print("Rows with no airline name:", df["name"].isna().sum())
print("Distinct carriers:", df["carrier"].nunique())

### Decision 2 — Whole-number columns stored as float → nullable Int64

In [ ]:
int_cols = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time"]
assert all((df[c].dropna() % 1 == 0).all() for c in int_cols)   # nothing is lost by the conversion
df[int_cols] = df[int_cols].astype("Int64")
print(df[int_cols].dtypes)

### Decision 3 — `time_hour`: text (UTC) → timezone-aware datetime in America/New_York

In [ ]:
t = pd.to_datetime(df["time_hour"], utc=True).dt.tz_convert("America/New_York")
print("Unparseable:", t.isna().sum())
agree = ((t.dt.year == df["year"]) & (t.dt.month == df["month"]) & (t.dt.day == df["day"])).sum()
print("Local date agrees with year/month/day:", f"{agree:,} of {len(df):,}")
df["time_hour"] = t

### Decision 4 — Clock times: `2400` means midnight → recode to `0` in new columns

In [ ]:
for c in ["dep_time", "arr_time"]:
    print(c, "== 2400:", int((df[c] == 2400).sum()))
    df[c + "_clock"] = df[c].mask(df[c] == 2400, 0)
print("Rows touched:", int(((df["dep_time"] == 2400) | (df["arr_time"] == 2400)).sum()))
print("sched_dep_time / sched_arr_time == 2400:", int((df["sched_dep_time"] == 2400).sum()), int((df["sched_arr_time"] == 2400).sum()))

### Decision 5 — Cancelled flights (no departure time): flag, do not drop

In [ ]:
df["cancelled"] = df["dep_time"].isna()
print("Cancelled:", int(df["cancelled"].sum()), f"({df['cancelled'].mean():.2%})")
print("All delay/air-time values missing for these rows:",
      bool(df.loc[df["cancelled"], ["dep_delay", "arr_time", "arr_delay", "air_time"]].isna().all().all()))

### Decision 6 — Missing `tailnum`: leave as missing

In [ ]:
miss_tail = df["tailnum"].isna()
print("Missing tailnum:", int(miss_tail.sum()), f"({miss_tail.mean():.2%})")
print("...of which cancelled:", int((miss_tail & df["cancelled"]).sum()))
print("...of which flew:", int((miss_tail & ~df["cancelled"]).sum()))

### Decision 7 — Flew but no arrival delay / air time (probably diverted): flag, do not drop

In [ ]:
df["no_arrival_delay"] = df["arr_delay"].isna() & ~df["cancelled"]
print("Flagged:", int(df["no_arrival_delay"].sum()), f"({df['no_arrival_delay'].mean():.2%})")
print("  has arr_time but no arr_delay:", int((df["no_arrival_delay"] & df["arr_time"].notna()).sum()))
print("  has no arr_time:", int((df["no_arrival_delay"] & df["arr_time"].isna()).sum()))
print("Reconciliation of arr_delay missing:", int(df["arr_delay"].isna().sum()), "=",
      int(df["cancelled"].sum()), "+", int(df["no_arrival_delay"].sum()))

### Decision 8 — Extreme and negative delays: keep

In [ ]:
dd = df["dep_delay"].astype("float")
print("dep_delay >= 180 min:", int((dd >= 180).sum()), "   >= 720 min:", int((dd >= 720).sum()), "   max:", dd.max())
# is the delay consistent with the clock times (modulo 24 h)? if so it is real, not a typing error
to_min = lambda x: (x // 100) * 60 + x % 100
gap = ((to_min(df["dep_time"].astype("float")) - to_min(df["sched_dep_time"]) - dd) % 1440).dropna()
print("dep_delay inconsistent with clock times:", int((gap != 0).sum()))
print("Early departures (dep_delay < 0):", int((dd < 0).sum()), "   early arrivals:", int((df["arr_delay"].astype("float") < 0).sum()))

### Decision 9 — Duplicates

In [ ]:
key = ["time_hour", "carrier", "flight", "origin", "dest", "sched_dep_time"]
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicates on flight key:", int(df.duplicated(key).sum()))

### Decision 10 — Same route, different distance

In [ ]:
g = df.groupby(["origin", "dest"])["distance"]
bad = g.nunique()[lambda s: s > 1].index
print("Routes with more than one distance:", len(bad))
mode = g.transform(lambda s: s.mode().iloc[0])
off = (df["distance"] != mode)
print("Rows that differ from the route's usual distance:", int(off.sum()), "  max gap (miles):", int((df["distance"] - mode).abs().max()))
print(df[df.set_index(["origin", "dest"]).index.isin(bad)].groupby(["origin", "dest"])["distance"].value_counts())

### Decision 11 — Implausible ground speed (distance / air_time)

In [ ]:
mph = df["distance"] / (df["air_time"].astype("float") / 60)
print("Ground speed > 650 mph:", int((mph > 650).sum()), "   max:", round(mph.max(), 1))
print(df.loc[mph > 650, ["origin", "dest", "distance", "air_time"]])
print("Ground speed < 150 mph:", int((mph < 150).sum()), "(all short hops?)  longest distance among them:", int(df.loc[mph < 150, "distance"].max()))

### Decision 12 — Categorical codes

In [ ]:
for c in ["carrier", "origin", "dest", "tailnum", "name"]:
    s = df[c].dropna()
    print(c, "| distinct:", s.nunique(), "| leading/trailing spaces:", int((s != s.str.strip()).sum()),
          "| not upper-case:", int((s != s.str.upper()).sum()) if c != "name" else "n/a")
print("Constant columns:", [c for c in df.columns if df[c].nunique(dropna=False) == 1])

### Result

In [ ]:
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   (working file had 20; +4 derived/flag columns)")
analysis_dep = df[~df["cancelled"]]                         # use for dep_delay questions
analysis_arr = df[~df["cancelled"] & ~df["no_arrival_delay"]]   # use for arr_delay / air_time questions
print("dep_delay analysis rows:", f"{len(analysis_dep):,}", "  arr_delay analysis rows:", f"{len(analysis_arr):,}")